## Problem 1: Representing SHA-256 Data

#### Representing 32-bit words

Research:
- Section 2 of FIPS 180-4 defines a WORD as a string of 32 bits for SHA-256.
- Section 3.2 defines addition on words as modulo 2^32, so my representation has to wrap around on overflow.
- A Python `int` has no fixed width, so I would need to mask with `0xFFFFFFFF` after every addition or left shift. Which would require alot of extra work.
- The NumPy docs - (link 1) say NumPy integers have a fixed size and overflow when a value doesn't fit, unlike a Python `int`.

Choice: `np.uint32` (link 2) because it is fixed at 32 bits, so results wrap modulo 2^32 without manual masking.

1. https://numpy.org/doc/stable/user/basics.types.html#overflow-errors
2. https://numpy.org/devdocs/reference/arrays.scalars.html#numpy.uint32

In [ ]:
## Code work for Representing 32-bit words

import numpy as np  # load NumPy so we can use its fixed-size integer types

# Make a 32-bit word holding the biggest value that fits (all 32 bits are 1)
word = np.uint32(0xFFFFFFFF)
print(hex(word))              # show it in hex: 0xffffffff

# Turn off NumPy's overflow warning, because we want it to wrap
with np.errstate(over="ignore"):
    # Add 1 to the biggest value which results in it not fitting, so it wraps back round to 0
    print(hex(word + np.uint32(1)))   # shows 0x0

# Do the same sum with a normal Python int
# It just gets bigger (33 bits), so it is no longer a valid 32-bit word
print(hex(0xFFFFFFFF + 1))    # shows 0x100000000

0xffffffff
0x0
0x100000000


### Representing sequences of 32-bit words

Research:
- FIPS 180-4, Section 5.2.1 divides each 512-bit block into sixteen 32-bit words.
- Section 6.2.2 specifies a message schedule of 64 words and a hash value composed of 8 words.
- Every sequence required therefore has a fixed length, and every element is a 32-bit word.
- According to the NumPy documentation (link 1), an array stores items of the same type and size, so each element remains a 32-bit word.

Choice: use a NumPy array with dtype=np.uint32 (link 2), because it has a fixed length and every element is a 32-bit word that wraps in the same way as the individual words described above.

1. https://numpy.org/doc/stable/reference/arrays.ndarray.html
2. https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32